# Notebook 03: Exploratory Data Analysis (EDA)
## Univariate, Bivariate, Regional & Temporal Patterns across Formula 1 (2010–2019)

### Analytical Objectives:
1. Examine univariate distributions, central tendencies, and dispersions across sporting and commercial metrics.
2. Analyze longitudinal time-series trends across 10 championship seasons.
3. Compare regional attendance differences across Continents (Europe, Asia, Americas, Middle East, Oceania).
4. Benchmark Buddh International Circuit against global baseline quartiles and peer inaugural circuits.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

if not os.path.exists('data') and os.path.exists('../data'):
    os.chdir('..')

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
f1_df = pd.read_csv("data/processed/f1_races_clean.csv")
ind_df = pd.read_csv("data/processed/india_gp_clean.csv")
print(f"Loaded {len(f1_df)} baseline races and {len(ind_df)} Indian GP editions.")

### 1. Descriptive Summary Statistics of Baseline Numerical Metrics

In [ ]:
metrics = [
    'attendance_race_day', 'attendance_weekend', 'average_speed_kmh',
    'winning_margin_s', 'finishers_count', 'dnf_count', 'total_pit_stops',
    'circuit_length_km', 'gdp_per_capita_usd', 'event_tenure_years'
]

summary_stats = f1_df[metrics].describe().T
summary_stats['IQR'] = summary_stats['75%'] - summary_stats['25%']
summary_stats['Skewness'] = f1_df[metrics].skew()
summary_stats['Kurtosis'] = f1_df[metrics].kurtosis()

pd.set_option('display.float_format', lambda x: '%.2f' % x)
summary_stats[['mean', 'std', 'min', '50%', 'max', 'IQR', 'Skewness', 'Kurtosis']]

### 2. Longitudinal Season-over-Season Attendance Trends (2010–2019)

In [ ]:
season_trends = f1_df.groupby('season').agg(
    races=('race_id', 'count'),
    mean_sunday=('attendance_race_day', 'mean'),
    median_sunday=('attendance_race_day', 'median'),
    mean_weekend=('attendance_weekend', 'mean')
).reset_index()

print("Longitudinal Attendance Trends:")
print(season_trends)

### 3. Regional Breakdown (Continental Comparison)

In [ ]:
regional_summary = f1_df.groupby('continent').agg(
    races=('race_id', 'count'),
    mean_sunday=('attendance_race_day', 'mean'),
    median_sunday=('attendance_race_day', 'median'),
    std_sunday=('attendance_race_day', 'std'),
    mean_gdp=('gdp_per_capita_usd', 'mean')
).reset_index().sort_values(by='mean_sunday', ascending=False)

print("Regional Summary Statistics:")
print(regional_summary)

### 4. Indian Grand Prix Benchmarking vs Global Baseline

In [ ]:
global_p25 = f1_df['attendance_race_day'].quantile(0.25)
global_p50 = f1_df['attendance_race_day'].quantile(0.50)
global_p75 = f1_df['attendance_race_day'].quantile(0.75)

print(f"Global Sunday Attendance Quartiles: 25th={global_p25:,.0f} | 50th={global_p50:,.0f} | 75th={global_p75:,.0f}\n")

for _, r in ind_df.iterrows():
    att = r['attendance_race_day']
    pctile = (f1_df['attendance_race_day'] < att).mean() * 100
    print(f"India GP {r['season']}: Sunday Attendance = {att:,} -> Baseline Percentile = {pctile:.1f}th")